**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/encorescience/encore-registry", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/elliott-2026"], check=True)
    os.chdir("/content/encore/entries/elliott-2026")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Reproducing Elliott, Andrews et al. (2026): claims on the raw voltage

Paper: Elliott, Andrews et al., "Microscale maps of bursting dynamics across human hippocampal slices
from patients with epilepsy", J. Neurophysiol. (2026), doi:10.1152/jn.00217.2025; claims file
`claims/elliott-2026.yaml`, written from the accepted manuscript (PMC12931417) and its Extended Data.

Dataset: DANDI:001132 version 0.241130.1903 (the most recent published version; the paper cites no
version, and the two published versions list the same 69 assets; see the header of
`reproduce_elliott_2026.ipynb`). Raw voltage is streamed; spike sortings come from the paper's
repository at commit 0f83f70 (outside input). The link between sortings and raw files is
established in `reproduce_elliott_2026.ipynb` (claim sorting-dandiset-link) and repeated here in
short: recordings 1 to 6 are found in the dandiset, recordings 7 (S5) and 8 (S6) are not.

| Claim | Status here |
|---|---|
| table2-phase-locking-counts | tested on recordings 1 to 6; 7 and 8 have no raw recording |
| table2-phase-locking-seizure-vs-not | tested |
| fig2b-phase-cluster-kuiper | tested |
| ext3-onset-spectral-signature | tested |
| ext4-electrode-lag-clusters | tested on S1 to S4; S5 and S6 have no raw recording |
| fig1j-theta-propagation-stages | tested |
| ext4b-theta-delay | tested (one of two readings) |
| fig2a-subbursts-track-theta | tested |
| methods-rolling-wave-counts | blocked (methods_underspecified); the criterion's detector is run and shown |

In [ ]:
import re, time, zlib, resource
import multiprocessing as mp
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from concurrent.futures import ProcessPoolExecutor
from scipy import stats, signal
from scipy.ndimage import gaussian_filter
from sklearn.cluster import KMeans
from astropy.stats import kuiper_two
import hippo as H
T0 = time.time()
POOL = ProcessPoolExecutor(4, mp_context=mp.get_context('fork'))  # for hippo's module-level readers
def pmap(fn, items):
    # a fresh forked pool, so that functions and data defined in the notebook are seen by the workers
    with ProcessPoolExecutor(4, mp_context=mp.get_context('fork')) as p:
        return list(p.map(fn, items))
print('dandiset', H.DANDISET_ID, 'version', H.DANDISET_VERSION, '; code commit', H.CODE_COMMIT)

## Data check

Loads the sortings and the electrode tables of the candidate files, and finds each sorting's file
and offset with the same scan as the spike notebook (only among files whose electrode layout holds
every unit position).

In [ ]:
assets = H.list_assets()
A = {a['path'].split('/')[0]: a for a in assets}
SORT = {r: H.load_sorting(r) for r in H.RECORDINGS}
def elec_job(sub):
    f = H.open_nwb(A[sub]['asset_id'])
    d = f['acquisition/ElectricalSeries/data']
    return sub, H.electrodes(f), d.shape, float(d.attrs['conversion'])
ELEC, SHAPE, CONV = {}, {}, {}
for sub, e, shp, cv in pmap(elec_job, sorted(A)):
    ELEC[sub], SHAPE[sub], CONV[sub] = e, shp, cv
cand = {}
for r, s in SORT.items():
    P = set(map(tuple, np.round(s['pos'], 1)))
    cand[r] = [sub for sub, e in ELEC.items() if P <= set(zip(np.round(e.x, 1), np.round(e.y, 1)))]
    print(r, H.RECORDINGS[r]['slice'], H.RECORDINGS[r]['cond'], s['n'], 'units,', f"{s['length_ms']/60000:.2f} min sorted; layout matches", cand[r])

In [ ]:
t = time.time()
def scan(job):
    r, sub = job
    return (r, sub) + H.offset_scan(A[sub]['asset_id'], SORT[r])
SCAN = pd.DataFrame(list(pmap(scan, [(r, sub) for r in SORT for sub in cand[r]])), columns=['rec', 'file', 'offset_s', 'peak', 'chance'])
SCAN['ratio'] = SCAN.peak / SCAN.chance
LINK = {}
for r in SORT:
    d = SCAN[SCAN.rec == r]; hit = d[d.ratio > 10]
    if len(hit) == 1:
        LINK[r] = (hit.file.iloc[0], round(float(hit.offset_s.iloc[0]), 3))
    print(r, H.RECORDINGS[r]['slice'], H.RECORDINGS[r]['cond'], '->', LINK.get(r, 'no file matched'), f'best ratio {d.ratio.max():.1f}')
for r, (sub, off) in LINK.items():
    e = ELEC[sub]
    print(f"{sub}: {SHAPE[sub][1]} channels, {SHAPE[sub][0]/H.FS:.1f} s at {H.FS:.0f} Hz, {CONV[sub]*1e6:.3f} uV per count; "
          f"electrodes x {e.x.min():.0f}-{e.x.max():.0f}, y {e.y.min():.0f}-{e.y.max():.0f} um; sorting spans file time {off:.1f} to {off + SORT[r]['length_ms']/1000:.1f} s")
print(f'{time.time()-t:.0f} s')

## LFP at the units' electrodes

For recordings 1 to 6, the 1 kHz LFP (`hippo.lfp_1khz`: 20-sample means of the raw voltage) of the
electrode nearest each unit, over the whole span of the sorting, on the sorting's clock (file time
minus the offset found above). Every unit position is an electrode position (data check), so the
nearest electrode is the unit's own best channel.

In [ ]:
t = time.time()
ULFP, UCH, UCOL = {}, {}, {}
for r, (sub, off) in LINK.items():
    e = ELEC[sub]
    ch = np.array([int(np.argmin((e.x - x) ** 2 + (e.y - y) ** 2)) for x, y in SORT[r]['pos']])
    uch = np.unique(ch)
    X = H.lfp_1khz(A[sub]['asset_id'], uch, off, off + SORT[r]['length_ms'] / 1000, len(e), POOL)
    ULFP[r] = X; UCH[r] = ch; UCOL[r] = {c: i for i, c in enumerate(uch)}
    dist = np.hypot(e.x.values[ch] - SORT[r]['pos'][:, 0], e.y.values[ch] - SORT[r]['pos'][:, 1])
    print(r, sub, X.shape, f'max unit-electrode distance {dist.max():.1f} um; LFP range {X.min():.0f} to {X.max():.0f} uV; {time.time()-t:.0f} s')
print(f'peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss/1e6:.1f} GB')

## Values taken from the paper's code

The seizure and non-seizure windows, the spans of the other recordings, the sub-burst times and the
phase-cluster thresholds are set by hand in the paper's code (claims file). They are defined here
and printed beside their source at the pinned commit: the windows and spans are the start and stop
times in the names of the LFP files the code saved (`data/ephys_raw/`; the S1 windows also appear,
commented out, in `TJ_phase_locking_basics.ipynb`, and the S2 windows there as active code); the
spans of recordings 2 and 5 include files whose times are not in the saving notebooks
(`get_raw_data/hc112823_base.ipynb` lists four of the five S1 baseline spans, `hc111.ipynb` one of
the three S3 spans). The code filters theta within each saved window (the `butter_filter` line
printed below), as is done here.

In [ ]:
SEIZ = {1: [(9,28),(38,52),(69,83),(100,113),(138,151),(176,187),(211,220),(246,254)],
        3: [(0,11),(25,39),(139,148),(168,174),(258,263),(320,328),(481,487)]}
NONS = {1: [(28,38),(54,66),(85,97),(115,127),(155,167),(193,205),(228,240),(260,272)],
        3: [(50,60),(100,113),(200,212),(270,275),(392,400),(400,406),(450,455)]}
SPANS_FILE = {2: [(25,35),(60,70),(105,115),(150,160),(250,260)], 4: [(60,70),(100,110),(150,160),(200,210),(250,260)],
              5: [(34,44),(200,210),(325,335)], 6: [(33,43),(110,120),(145,155)], 7: [(120,130),(190,200),(350,365)], 8: [(6,16),(19,29),(50,60)]}
SUBB = {1: [14.4, 14.57, 14.75, 14.91], 3: [35.208, 35.234, 35.265, 35.302]}
def code_group1(pos):
    x, y = pos[:, 0], pos[:, 1]
    return (x < 940) | ((x == 945) & (y < 1120)) | ((x > 900) & (y < 900))

tree = H.code_tree()
RAW = 'data/ephys_raw/'
sources = {(1, 'seizure'): '2023-12-03-e-Hc112823_avv9hckcr1/seizure/', (1, 'nonseizure'): '2023-12-03-e-Hc112823_avv9hckcr1/nonseizure/',
           (3, 'seizure'): '2023-12-11-e-Hc120523_aav9hckcr1_chip21769/seizure/', (3, 'nonseizure'): '2023-12-11-e-Hc120523_aav9hckcr1_chip21769/nonseizure/',
           (2, 'spans'): '2023-12-03-e-Hc112823_avv9hckcr1/baseline/base_', (4, 'spans'): '2023-12-11-e-Hc120523_aav9hckcr1_chip21769/baseline/base_',
           (5, 'spans'): '2022-11-02-e-Hc11.1-chip16753/immediate_', (6, 'spans'): '2023-11-13-e-Hc110723_hckcr1_21841/acid1_',
           (7, 'spans'): '2023-04-02-e-hc328_unperturbed/raw_', (8, 'spans'): '2023-05-10-e-hc52_18790_unperturbed/raw_'}
used = {**{(r, 'seizure'): w for r, w in SEIZ.items()}, **{(r, 'nonseizure'): w for r, w in NONS.items()}, **{(r, 'spans'): w for r, w in SPANS_FILE.items()}}
for k, pre in sources.items():
    files = sorted(p[len(RAW):] for p in tree if p.startswith(RAW + pre) and '/' not in p[len(RAW + pre):] and p.endswith('_fs1ms.pkl'))
    found = sorted(tuple(int(v) for v in re.search(r'start(\d+)s_stop(\d+)s', f).groups()) for f in files)
    print(k, '| files:', [f.split('/')[-1] for f in files], '| used here equal:', found == sorted(used[k]))
TJ = 'figures/figure_3/TJ_phase_locking_basics.ipynb'
lines = H.code_lines(TJ, r'^#?\s*\[\d+, \d+\]')
act = [tuple(int(v) for v in re.findall(r'\d+', l)) for _, _, l in lines if not l.startswith('#')]
com = [tuple(int(v) for v in re.findall(r'\d+', l)) for _, _, l in lines if l.startswith('#')]
print(f'{TJ} cell {lines[0][0]}: active windows {act}; equal to the S2 seizure and non-seizure windows: {act == SEIZ[3] + NONS[3]}')
print(f'  commented windows {com}; the first 16 equal to the S1 seizure and non-seizure windows: {com[:16] == SEIZ[1] + NONS[1]}')
for h in H.code_lines(TJ, r'theta_filt = butter_filter|NUM_CLOSEST =|RMS_MULT ='):
    print(f'  cell {h[0]} line {h[1]}: {h[2]}')
for r, path in {1: 'figures/figure_2/burst_peak_comparison.ipynb', 3: 'figures/supplement/seizure_2/seizure2_burst_peaks.ipynb'}.items():
    hits = H.code_lines(path, r'subtime')
    for h in hits:
        print(f'{path} cell {h[0]} line {h[1]}: {h[2]}')
    vals = sorted({float(v) for _, _, l in hits for v in re.findall(r'(\d+\.\d+)\s*\*\s*1000', l)})
    print('  sub-burst edges used here', SUBB[r], '| equal:', vals == SUBB[r])
h = H.code_lines('figures/figure_3/Phase_lock_heatmap.ipynb', r'sig_x<940')[0]
print(f'figures/figure_3/Phase_lock_heatmap.ipynb cell {h[0]} line {h[1]}: {h[2]}')

## table2-phase-locking-counts

> Significantly theta phase-locked units are 70 of 212 (33%) in the S1 KA recording and 101 of 180
> (56%) in the S2 KA recording, against 14 of 249, 2 of 51, 8 of 87, 6 of 109, 8 of 136 and 1 of 100
> in the other recordings.

Theta 4 to 8 Hz (order-5 Butterworth, zero phase), Hilbert phase and envelope, per window (as the code
filters each saved window, printed above); Rayleigh
test p < 0.05 per unit (astropy). Spike selection: the Methods (envelope above the RMS of the theta
series), with the RMS taken over the analysed windows (`envelope`) and over the whole recording
(`envelope_whole_rms`, the Methods' "RMS of the whole time series", the criterion's wording), and
the code's cycle selection as a variant. Over the whole recording the two envelope readings are
the same. Recordings 1 and 3:
the code's seizure windows. Others: the whole recording, and the code's get_raw_data spans as a
second reading. Those spans are in the raw file's time (the S4 span 145 to 155 s lies beyond the
130 s sorting and inside the 160 s file), so they are moved to the sorting's clock by the offset
found above; parts outside the sorting are dropped.

In [ ]:
def spans_sorting(r):
    off = LINK[r][1]; L = SORT[r]['length_ms'] / 1000
    out = [(max(a - off, 0), min(b - off, L)) for a, b in SPANS_FILE[r]]
    return [(a, b) for a, b in out if b - a > 1]

def lock_job(args):
    r, wins, mode = args
    X = ULFP[r]; res = []
    cache = {}
    for u in range(SORT[r]['n']):
        c = UCOL[r][UCH[r][u]]
        if c not in cache:
            th = [(a * 1000, H.theta(X[int(a * 1000):int(b * 1000), c])) for a, b in wins]
            # the Methods' "RMS of the whole time series": the theta of the whole recording at this electrode
            rms = float(np.sqrt(np.mean(H.theta(X[:, c]) ** 2))) if mode == 'envelope_whole_rms' else None
            cache[c] = (th, rms)
        th, rms = cache[c]
        res.append(H.phase_lock_unit(SORT[r]['trains'][u], th, 'envelope' if mode == 'envelope_whole_rms' else mode, rms))
    return pd.DataFrame(res, columns=['n_spikes', 'p', 'mean_angle'])

jobs = {}
for r in LINK:
    sets = {'whole': [(0, SORT[r]['length_ms'] / 1000)]}
    if r in SEIZ:
        sets.update(seizure=SEIZ[r], nonseizure=NONS[r])
    else:
        sets['code_spans'] = spans_sorting(r)
    for name, wins in sets.items():
        for mode in ('envelope', 'envelope_whole_rms', 'cycle'):
            jobs[(r, name, mode)] = (r, wins, mode)
t = time.time()
LOCK = dict(zip(jobs, pmap(lock_job, list(jobs.values()))))
rows = []
for (r, name, mode), d in LOCK.items():
    rows.append(dict(rec=r, slice=H.RECORDINGS[r]['slice'], cond=H.RECORDINGS[r]['cond'], span=name, selection=mode, units=len(d),
                     significant=int((d.p < 0.05).sum()), percent=100 * (d.p < 0.05).mean(), units_no_spikes=int((d.n_spikes < 2).sum()),
                     median_spikes_kept=float(d.n_spikes.median())))
PL = pd.DataFrame(rows)
print(PL.round(1).to_string(index=False)); print(f'{time.time()-t:.0f} s')
print('spans used (sorting clock):', {r: spans_sorting(r) for r in LINK if r not in SEIZ})

In [ ]:
exp_pl = {1: 70, 2: 14, 3: 101, 4: 2, 5: 8, 6: 6, 7: 8, 8: 1}
tol_pl = {1: 7, 2: 1.4, 3: 10.1, 4: 1, 5: 1, 6: 1, 7: 1, 8: 1}
rows = []
for mode in ('envelope', 'envelope_whole_rms', 'cycle'):
    for reading in ('whole', 'code_spans'):
        row = dict(selection=mode, reading_others=reading)
        n_ok = 0
        for r in LINK:
            span = 'seizure' if r in SEIZ else reading
            v = int(PL[(PL.rec == r) & (PL.span == span) & (PL.selection == mode)].significant.iloc[0])
            ok = abs(v - exp_pl[r]) <= tol_pl[r]
            row[f'r{r}'] = f"{v}{'*' if ok else ''}"
            n_ok += ok
        row['within_tol_of_6'] = n_ok
        rows.append(row)
print('count per recording (* within tolerance); recordings 7 and 8 have no raw recording in the dandiset')
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
# null: the same test with each unit's spike times shifted circularly by a random amount within its windows
rng = np.random.default_rng(zlib.crc32(b"elliott-2026/table2-phase-locking-counts"))
def shifted(r, wins, shift_ms):
    out = []
    for u, tr in enumerate(SORT[r]['trains']):
        parts = []
        for a, b in wins:
            x = tr[(tr >= a * 1000) & (tr < b * 1000)]
            parts.append(a * 1000 + np.mod(x - a * 1000 + shift_ms[u], (b - a) * 1000))
        out.append(np.sort(np.concatenate(parts)) if parts else tr)
    return out
for r in (1, 3):
    sh = rng.uniform(1000, 5000, SORT[r]['n'])
    tr_s = shifted(r, SEIZ[r], sh)
    X = ULFP[r]; cnt = 0
    cache = {}
    for u in range(SORT[r]['n']):
        c = UCOL[r][UCH[r][u]]
        if c not in cache:
            cache[c] = [(a * 1000, H.theta(X[int(a * 1000):int(b * 1000), c])) for a, b in SEIZ[r]]
        cnt += H.phase_lock_unit(tr_s[u], cache[c], 'envelope')[1] < 0.05
    print(H.RECORDINGS[r]['slice'], 'seizure windows, spikes shifted 1 to 5 s within each window:', int(cnt), 'of', SORT[r]['n'], 'significant (chance level 5% =', round(0.05 * SORT[r]['n'], 1), ')')

**Result: not reproduced.** With the Methods' spike selection the counts are 60 (S1 KA, seizure windows, 70 expected), 104 (S2 KA, 101), and for recordings 2, 4, 5 and 6 19, 5, 11 and 3 over the whole recording or 12, 0, 10 and 5 over the code's spans (14, 2, 8 and 6 expected): 1 or 2 of the 6 testable counts are within tolerance. With the RMS taken over the whole recording (the Methods' "RMS of the whole time series") the counts are 59 (S1) and 104 (S2), and 19, 5, 11 and 3 or 12, 0, 8 and 5 for the others: 1 or 3 of 6 within tolerance, and S1 misses under every envelope reading. With the code's cycle selection (variant) S1 and S2 match (73 and 107) and 3 to 4 of 6 are within tolerance; over the code's spans every other recording is below 10% (5.5 to 8.0%). Recordings 7 and 8 have no raw recording in the dandiset. Shifting spikes within the seizure windows gives 10 of 212 and 15 of 180, near the 5% chance level.

## table2-phase-locking-seizure-vs-not

> During seizure-like activity on average 45% of units per recording are phase locked, against 5% in
> recordings without seizure-like activity; within S1 and S2, only 5% and 4% are phase locked
> outside the seizure periods.

The mean over recordings 1 and 3 (seizure windows), the mean over the recordings without
seizure-like activity that have raw data here (2, 4, 5, 6; 7 and 8 are missing), and the
non-seizure windows of recordings 1 and 3.

In [ ]:
rows = []
for mode in ('envelope', 'envelope_whole_rms', 'cycle'):
    for reading in ('whole', 'code_spans'):
        g = lambda r, span: float(PL[(PL.rec == r) & (PL.span == span) & (PL.selection == mode)].percent.iloc[0])
        seiz = np.mean([g(1, 'seizure'), g(3, 'seizure')])
        other = np.mean([g(r, reading) for r in (2, 4, 5, 6)])
        n1, n3 = g(1, 'nonseizure'), g(3, 'nonseizure')
        vals = [seiz, other, n1, n3]; exp = [45, 5, 5, 4]
        ok = [abs(v - x) <= 0.1 * x for v, x in zip(vals, exp)]
        rows.append(dict(selection=mode, reading_others=reading, seizure_mean=seiz, others_mean=other, S1_nonseizure=n1, S2_nonseizure=n3,
                         n_within_tol=sum(ok), all_others_below_10=all(v < 10 for v in vals[1:]), seizure_ok=ok[0]))
print(pd.DataFrame(rows).round(2).to_string(index=False))

**Result: partially reproduced.** The seizure mean is 43.0% with the Methods' selection (within 40.5 to 49.5) and 46.9% with the code's. Outside the seizure windows 5.2% (S1, within tolerance) and 4.4% (S2, 8 of 180 units, just above the 4.4% bound) are phase locked. The mean over recordings 2, 4, 5 and 6 is 8.2% over whole recordings and 5.2% over the code's spans (within tolerance), so all four values match under no reading, and every value other than the seizure mean is below 10%. With the RMS over the whole recording the values are 42.8%, 8.2% or 4.7%, 5.7% and 3.9%: 2 or 3 of 4 within tolerance, every value other than the seizure mean below 10%.

## fig2b-phase-cluster-kuiper

> In S1 the significantly phase-locked units form two spatial groups along the granule cell layer
> whose preferred theta phases differ significantly (Kuiper test).

Units significant in the seizure windows (main) and over the whole KA recording (variant), each
under the Methods' and the code's spike selection. Splits: (a) the code's position thresholds, with
duplicate positions removed as the code does; (b) the angular difference from the polar mode split
at its median (polar mode from a 36-bin histogram, a standard choice); (c) k-means with 2 clusters
on unit position. Kuiper two-sample test (astropy kuiper_two).

In [ ]:
seed = zlib.crc32(b"elliott-2026/fig2b-phase-cluster-kuiper")
rows = []
for span in ('seizure', 'whole'):
    for mode in ('envelope', 'envelope_whole_rms', 'cycle'):
        d = LOCK[(1, span, mode)]; sig = np.where(d.p < 0.05)[0]
        ang = d.mean_angle.values[sig]; pos = SORT[1]['pos'][sig]
        h, ed = np.histogram(np.mod(ang, 2 * np.pi), 36, (0, 2 * np.pi)); mode_ang = (ed[h.argmax()] + ed[h.argmax() + 1]) / 2
        diff = np.abs(np.angle(np.exp(1j * (ang - mode_ang))))
        _, first = np.unique(pos, axis=0, return_index=True); dedup = np.zeros(len(sig), bool); dedup[first] = True
        splits = {'a_code_thresholds': (code_group1(pos), dedup),
                  'b_median_angular_difference': (diff > np.median(diff), np.ones(len(sig), bool)),
                  'c_kmeans_position': (KMeans(2, n_init=10, random_state=seed).fit_predict(pos) == 1, np.ones(len(sig), bool))}
        for name, (g, keep) in splits.items():
            g1, g2 = ang[g & keep], ang[~g & keep]
            D, p = kuiper_two(np.mod(g1, 2 * np.pi), np.mod(g2, 2 * np.pi)) if len(g1) > 1 and len(g2) > 1 else (np.nan, np.nan)
            m1 = np.degrees(np.angle(np.mean(np.exp(1j * g1)))); m2 = np.degrees(np.angle(np.mean(np.exp(1j * g2))))
            rows.append(dict(units=span, selection=mode, n_sig=len(sig), split=name, n1=len(g1), n2=len(g2),
                             mean_angle1_deg=m1, mean_angle2_deg=m2, kuiper_D=D, p=p))
K2 = pd.DataFrame(rows)
print(K2.round(4).to_string(index=False))

**Result: reproduced.** With the 60 units significant in the seizure windows (Methods' selection), the Methods' split by angular difference from the polar mode (reading b) gives Kuiper p below 1e-4; by construction this split separates the angles it then tests. The spatial splits give p = 0.037 (code's thresholds, 32 and 26 units after removing duplicate positions) and p = 0.11 (k-means on position). With the code's spike selection the spatial splits give p = 0.0042 and 0.0014, and with units significant over the whole recording p = 0.035 to 0.051 (Methods) and 0.0008 to 0.013 (code). With the 59 units significant in the seizure windows under the RMS of the whole recording, the spatial splits give p = 0.023 and 0.020, and reading b again p below 1e-4. Under the Methods' spike selection only reading b reaches p <= 0.01.

## All-electrode windows

Stage windows (claims file: baseline before the first burst, initiation the first burst, seizure the
subsequent bursts; exact windows not given). Bursts start within the first second of both KA
recordings (spike notebook), so no window lies before the first burst; the baseline is taken as a
stretch free of detected bursts under every burst reading, just before the initiation burst. S1
initiation is the first burst that the code splits into sub-bursts, 14.4 to 14.91 s, and the seizure
plot of Extended Data Fig. 4e starts at 15.10 s; S2 initiation is the code's sub-bursts at 35.208
to 35.302 s (figures/supplement/seizure_2/seizure2_burst_peaks.ipynb, printed above), and Extended
Data Fig. 5b marks seizure after it. Used here: S1 baseline 12.0 to 13.0 s, initiation 14.4 to 14.91
s, seizure 15.0 to 28.0 s (to the end of the code's first seizure window); S2 baseline 34.45 to 35.2
s (0.75 s: detected bursts at 34.35 to 34.42 s and 35.24 s bound it), initiation 35.208 to 35.302 s,
seizure 35.3 to 38.5 s. The code's S2 initiation window (94 ms) is shorter than one theta cycle
(125 to 250 ms), so a lag of up to 40 ms has little signal to rest on there; windows of one theta
cycle at 4 Hz and longer starting at 35.208 s (250 ms, 500 ms, 1 s) and 35.0 to 35.5 s are run as
variants. For S3 and S4 (Extended Data Fig. 6: segments "that contained higher than average
burst-like activity"; the length is not given) the 8 s window with the most spikes of the sorting
(8 s is a choice). S5 and S6 have no raw recording here.

In [ ]:
def busiest(r, w=8.0):
    allsp = np.concatenate(SORT[r]['trains']) / 1000
    c = np.histogram(allsp, bins=np.arange(0, SORT[r]['length_ms'] / 1000, 1.0))[0]
    k = np.convolve(c, np.ones(int(w)), 'valid').argmax()
    return float(k), float(k + w), c[k:k + int(w)].sum() / w / SORT[r]['n'], c.sum() / len(c) / SORT[r]['n']
WIN = {1: dict(read=(0, 30), baseline=(12.0, 13.0), initiation=(14.4, 14.91), seizure=(15.0, 28.0)),
       3: dict(read=(33, 40), baseline=(34.45, 35.2), initiation=(35.208, 35.302), seizure=(35.3, 38.5),
               init_250ms=(35.208, 35.458), init_500ms=(35.208, 35.708), init_1s=(35.208, 36.208), init_35_to_35p5=(35.0, 35.5))}
# bursts of S1 and S2 under the four readings of fig1g (as in the spike notebook)
READINGS = [(sg, rd) for rd in ('a', 'b') for sg in (10, 20)]
def bursts(r, sg, rd):
    s = SORT[r]; rate = H.population_rate(s['trains'], s['length_ms'], sg)
    if rd == 'a':
        b = {1: 2, 3: 4}[r]
        thr = 2 * H.population_rate(SORT[b]['trains'], SORT[b]['length_ms'], sg).mean()
    else:
        thr = rate.mean() + 3 * rate.std()
    return H.detect_bursts(rate, thr)
BT = {(r, sg, rd): bursts(r, sg, rd) for r in (1, 3) for sg, rd in READINGS}
for r in (1, 3):
    lo, hi = WIN[r]['baseline'][0] - 2, WIN[r]['initiation'][1]
    for sg, rd in READINGS:
        b = BT[(r, sg, rd)][0]
        inb = b[(b >= WIN[r]['baseline'][0]) & (b < WIN[r]['baseline'][1])]
        print(H.RECORDINGS[r]['slice'], f'({rd}) sigma {sg}: bursts {lo:.2f} to {hi:.2f} s: {np.round(b[(b >= lo) & (b < hi)], 3).tolist()}; in the baseline window: {len(inb)}')
for r in (5, 6):
    a, b, act, mean_ = busiest(r); off = LINK[r][1]
    WIN[r] = dict(read=(a + off - 1, b + off + 1), active=(a + off, b + off))
    print(H.RECORDINGS[r]['slice'], f'busiest 8 s: sorting time {a:.0f}-{b:.0f} s (file {a+off:.0f}-{b+off:.0f} s), {act:.2f} spikes/s per unit against {mean_:.2f} over the recording')
t = time.time()
WL = {}
for r, w in WIN.items():
    sub = LINK[r][0]; e = ELEC[sub]
    WL[r] = H.lfp_1khz(A[sub]['asset_id'], np.arange(len(e)), w['read'][0], w['read'][1], len(e), POOL)
    print(r, sub, WL[r].shape, f'range {WL[r].min():.0f} to {WL[r].max():.0f} uV; {time.time()-t:.0f} s')

## ext4-electrode-lag-clusters

> In the two epileptiform slices, k-means clustering of electrodes by their pairwise theta lags gives
> spatial clusters that replicate the theta propagation of each stage, while the slices without
> epileptiform activity (S3 to S6) show no spatial pattern.

Lag matrix as the Methods (`hippo.lag_matrix`), k-means with 2 clusters on its rows (random state
from the claim's seed), spatial coherence against 1000 label shuffles (`hippo.spatial_coherence`).
Theta filter order 5 (code) and 4 (Methods' theta plots). The S2 initiation window of the code is
94 ms, shorter than one theta cycle; the longer windows above are run as variants, with the fraction
of lags at the +-40 ms edge (`frac_at_edge`).

In [ ]:
seed = zlib.crc32(b"elliott-2026/ext4-electrode-lag-clusters")
LAGS, LABS = {}, {}
rows = []
for r, w in WIN.items():
    e = ELEC[LINK[r][0]]; xy = np.c_[e.x, e.y]; t0 = w['read'][0]
    for order in (5, 4):
        th = H.theta(WL[r], order=order)
        for st, (a, b) in w.items():
            if st == 'read':
                continue
            seg = th[int(round((a - t0) * 1000)):int(round((b - t0) * 1000))]
            L, Cc = H.lag_matrix(seg)
            lab = KMeans(2, n_init=10, random_state=seed).fit_predict(L)
            obs, null = H.spatial_coherence(xy, lab, rng=np.random.default_rng(seed))
            LAGS[(r, order, st)] = L; LABS[(r, order, st)] = lab
            rows.append(dict(slice=H.RECORDINGS[r]['slice'], order=order, stage=st, window_ms=len(seg), sizes=np.bincount(lab).tolist(),
                             coherence=obs, null_mean=null.mean(), null_max=null.max(), beyond_all=obs > null.max(),
                             median_abs_lag=float(np.median(np.abs(L))), frac_at_edge=float(np.mean(np.abs(L) == 40))))
LC = pd.DataFrame(rows)
print(LC.round(3).to_string(index=False))

In [ ]:
# seed sensitivity of the k-means labels and the shuffle null (claim seed plus 1000 to 4000)
rows = []
for r, st in [(1, 'initiation'), (1, 'seizure'), (3, 'initiation'), (3, 'init_250ms'), (3, 'init_500ms'), (3, 'init_1s'), (3, 'init_35_to_35p5'), (3, 'seizure'), (5, 'active'), (6, 'active')]:
    e = ELEC[LINK[r][0]]; xy = np.c_[e.x, e.y]; L = LAGS[(r, 5, st)]
    for k in range(5):
        sd = seed + 1000 * k
        lab = KMeans(2, n_init=10, random_state=sd).fit_predict(L)
        obs, null = H.spatial_coherence(xy, lab, rng=np.random.default_rng(sd))
        rows.append(dict(slice=H.RECORDINGS[r]['slice'], stage=st, seed_offset=1000 * k, coherence=obs, null_max=null.max(), beyond_all=obs > null.max()))
print(pd.DataFrame(rows).round(3).to_string(index=False))

In [ ]:
fig, axs = plt.subplots(2, 4, figsize=(14, 6))
for a, (r, st) in zip(axs.flat, [(1, 'baseline'), (1, 'initiation'), (1, 'seizure'), (3, 'baseline'), (3, 'initiation'), (3, 'seizure'), (5, 'active'), (6, 'active')]):
    e = ELEC[LINK[r][0]]
    a.scatter(e.x, e.y, c=LABS[(r, 5, st)], s=3, cmap='PiYG'); a.invert_yaxis(); a.set_aspect('equal')
    a.set_title(f"{H.RECORDINGS[r]['slice']} {st}", fontsize=9)
plt.tight_layout(); plt.show()

**Result: partially reproduced.** Electrode clusters are spatially coherent beyond all 1000 shuffles in the S1 initiation (0.69 against a shuffle maximum of 0.57) and seizure stages (0.90 against 0.52) and the S2 seizure stage (0.86 against 0.48), under both filter orders and all five seeds. The S2 initiation window of the code (94 ms, shorter than one theta cycle; median lag 0 ms, 0.2% of lags at the window edge) gives 0.84 against 0.86 and fails; every window of one theta cycle or longer passes under both filter orders and all five seeds (250 ms from 35.208 s: 0.92 to 0.93 against shuffle maxima of 0.64 to 0.65; 500 ms: 0.88 against 0.45 to 0.48; 1 s: 0.88 against 0.45 to 0.48; 35.0 to 35.5 s: 0.91 against 0.75 to 0.76), with 11 to 22% of lags at the edge. S3 and S4 are less coherent (0.44 and 0.44, below their shuffle maxima of 0.49 and 0.48) than every coherent stage of S1 and S2. S5 and S6 could not be tested, so the second part of the criterion is shown for 2 of the 4 slices it names. The S1 baseline is not coherent (0.37 against 0.44), and the S2 baseline is (0.67 against 0.50), where the paper reports no pattern at baseline.

## fig1j-theta-propagation-stages

> In S1 theta-band activity shows no pattern at baseline, a standing wave oscillating across the
> length of the granule cell layer at initiation, and at the seizure stage a rolling wave orthogonal
> to the initiation direction, moving from the hilar to the outer aspect; S2 shows the same stages.

Per stage, a plane fitted to each electrode's arrival time (minus its mean lag) over position
(`hippo.plane_direction`). Layer long axis and the normal toward the convex (outer) side from the
degree-2 fit of unit positions (`hippo.layer_axes`; that the hilus lies on the concave side is an
assumption). Initiation direction is compared with the long axis as an axis (a standing wave has no
sign); seizure direction with the outward normal as a direction. `seiz_vs_inward` gives the angle if the hilus were on the convex side
instead. For S2 the initiation is also run with the 250 ms window, as the code's 94 ms window is
shorter than one theta cycle.

In [ ]:
rows = []
for r in (1, 3):
    ax_, nrm = H.layer_axes(SORT[r]['pos'])
    ax_deg = np.degrees(np.arctan2(ax_[1], ax_[0])); n_deg = np.degrees(np.arctan2(nrm[1], nrm[0]))
    e = ELEC[LINK[r][0]]; xy = np.c_[e.x, e.y]
    for order in (5, 4):
        for ini_stage in (['initiation'] if r == 1 else ['initiation', 'init_250ms']):
            d = {st: H.plane_direction(xy, LAGS[(r, order, st)]) for st in ('baseline', ini_stage, 'seizure')}
            ini, sz = d[ini_stage][0], d['seizure'][0]
            row = dict(slice=H.RECORDINGS[r]['slice'], order=order, init_window=ini_stage, layer_axis_deg=ax_deg, outward_normal_deg=n_deg,
                       init_dir=ini, init_R2=d[ini_stage][1], seiz_dir=sz, seiz_R2=d['seizure'][1], base_R2=d['baseline'][1],
                       init_vs_axis=H.angle_between(ini, ax_deg, axial=True), init_vs_seiz=H.angle_between(ini, sz, axial=True),
                       seiz_vs_outward=H.angle_between(sz, n_deg), seiz_vs_inward=H.angle_between(sz, n_deg + 180))
            row['holds'] = (row['init_vs_axis'] <= 45) and (60 <= row['init_vs_seiz'] <= 120) and (row['seiz_vs_outward'] <= 45) and (row['base_R2'] < min(row['init_R2'], row['seiz_R2']))
            rows.append(row)
print(pd.DataFrame(rows).round(3).to_string(index=False))

**Result: not reproduced.** Plane fits explain little of the arrival times (R2 0.02 at initiation and 0.05 in the seizure stage of S1; 0.02 and 0.31 in S2). The initiation direction is 75 to 79 degrees from the layer's long axis in both slices, the initiation and seizure directions differ by only 4 to 16 degrees, and the seizure direction points 151 to 173 degrees from the outward normal (7 to 29 degrees from the inward normal, if the hilus lay on the convex side). In S1 the baseline R2 (0.000 to 0.001) is the lowest of the three stages; in S2 it (0.06 to 0.07) is above that of the code's initiation window. With the 250 ms S2 initiation window the initiation direction lies 27 degrees from the long axis (R2 0.43), but it differs from the seizure direction by 34 degrees, outside 60 to 120. The conditions fail in S1 under every variant.

## ext4b-theta-delay

> During seizure-like activity theta waves in S1 reach three electrodes along the propagation
> direction in turn, with a 40 ms delay from the first to the second and from the second to the third.

Reading 1 of the criterion: the median lag over electrode pairs whose displacement lies within 15
degrees of the seizure-stage direction (from fig1j) at the spacing of the figure's electrodes. The
spacing is read from Extended Data Fig. 4a against its 500 um scale bar as roughly 200 to 300 um
between successive electrodes (an approximate reading of the figure). The second reading (the three
electrodes of the figure's map) cannot be run: the map is drawn over histology without electrode
coordinates.

In [ ]:
e = ELEC[LINK[1][0]]; xy = np.c_[e.x, e.y]
for order in (5, 4):
    L = LAGS[(1, order, 'seizure')]
    sdir = H.plane_direction(xy, L)[0]
    u = np.array([np.cos(np.radians(sdir)), np.sin(np.radians(sdir))])
    dvec = xy[None, :, :] - xy[:, None, :]
    dist = np.hypot(dvec[..., 0], dvec[..., 1])
    cosang = (dvec @ u) / np.maximum(dist, 1e-9)
    for lo, hi in [(200, 300), (150, 250), (250, 350)]:
        m = (dist >= lo) & (dist < hi) & (cosang > np.cos(np.radians(15)))
        lag = L[m]
        print(f'order {order}, direction {sdir:.0f} deg, spacing {lo}-{hi} um: {m.sum()} pairs, median lag {np.median(lag):.1f} ms, '
              f'IQR {np.percentile(lag, 25):.0f} to {np.percentile(lag, 75):.0f}, fraction positive {np.mean(lag > 0):.2f}, at +-40 ms {np.mean(np.abs(lag) == 40):.2f}')

**Result: not reproduced.** Along the fitted seizure direction at 200 to 300 um, the median lag is -23 ms (interquartile range -38 to -9 ms; 13% positive), and -19 to -23 ms at the other spacings, with 19 to 34% of lags at the +-40 ms edge. The negative sign means electrodes further along the fitted direction lead, consistent with the weak plane fit of fig1j. The reading with the figure's three electrodes could not be run.

## methods-rolling-wave-counts (blocked, methods_underspecified)

> The rolling wave appeared in every investigated burst of S1 and in about 75% of those of S2, 2 to
> 4 times per burst, giving about 200 stereotypic propagations in S1's seizure stage and 35 in S2's.

The paper counted rolling waves by watching videos. The criterion allows one detector as an
assumption: a theta cycle whose lag-plane direction matches the seizure direction (from
fig1j-theta-propagation-stages) within 45 degrees and explains most of the lag variance. It is run
here on the S1 seizure window (15 to 28 s): theta cycles from trough to trough of the theta averaged
over all electrodes, one lag matrix and plane fit per cycle (`hippo.lag_matrix`,
`hippo.plane_direction`), "most of the variance" read as R2 above 0.5. The claim stays blocked: the
reference direction itself rests on a plane fit with R2 0.05 (fig1j), so the detector has no
defensible basis (see the attempt record).

In [ ]:
t = time.time()
e = ELEC[LINK[1][0]]; xy = np.c_[e.x, e.y]; t0 = WIN[1]['read'][0]
th = H.theta(WL[1], order=5)
sdir = H.plane_direction(xy, LAGS[(1, 5, 'seizure')])[0]
tr = signal.find_peaks(-th.mean(1))[0]
tr = tr[(tr >= (15.0 - t0) * 1000) & (tr < (28.0 - t0) * 1000)]
rows = []
for a, b in zip(tr[:-1], tr[1:]):
    d, r2, _ = H.plane_direction(xy, H.lag_matrix(th[a:b])[0])
    rows.append(dict(start_s=t0 + a / 1000, ms=b - a, direction=d, R2=r2, off_seizure_dir=H.angle_between(d, sdir)))
RW = pd.DataFrame(rows)
RW['is_rolling'] = (RW.R2 > 0.5) & (RW.off_seizure_dir <= 45)
print(f'seizure direction {sdir:.0f} deg; {len(RW)} theta cycles of {RW.ms.min()} to {RW.ms.max()} ms; R2 median {RW.R2.median():.4f}, '
      f'max {RW.R2.max():.3f}; cycles with R2 > 0.5: {int((RW.R2 > 0.5).sum())}; within 45 deg of the seizure direction: {int((RW.off_seizure_dir <= 45).sum())}; '
      f'both (rolling waves): {int(RW.is_rolling.sum())}')
for sg, rd in READINGS:
    b = BT[(1, sg, rd)][0]; b = b[(b >= 15.0) & (b < 28.0)]
    with_wave = sum(RW[(RW.start_s >= x - 0.25) & (RW.start_s < x + 0.25)].is_rolling.any() for x in b)
    print(f'({rd}) sigma {sg}: {len(b)} bursts in 15 to 28 s, {with_wave} with a rolling wave within 250 ms')
print(f'{time.time() - t:.0f} s')

**Result: blocked (methods_underspecified).** The criterion's detector finds no rolling wave in S1's first seizure window: of 63 theta cycles (106 to 285 ms), none has a plane fit with R2 above 0.5 (median 0.0023, largest 0.38), though 37 point within 45 degrees of the fitted seizure direction; 0 of the 7 to 22 bursts in 15 to 28 s have a rolling wave. Adopting the detector would give 0 for every expected value; it is not adopted, as its reference direction rests on a plane fit with R2 0.05.

## fig2a-subbursts-track-theta

> At the initiation stage of S1 the first burst splits into three sub-bursts whose peaks and troughs
> coincide with the theta oscillation, and the change in single-unit firing from sub-burst to
> sub-burst alternates between the same two spatial modes as theta; S2 shows a similar pattern.

Sub-bursts as set in the code (S1 14.4, 14.57, 14.75, 14.91 s; S2 35.208, 35.234, 35.265, 35.302 s;
sources printed above).
Per-unit rate in each sub-burst, the difference between consecutive sub-bursts (earlier minus
later, as the code), heatmaps on a 30 by 30 grid with Gaussian smoothing (Methods; display only).
Theta modes: the two k-means electrode clusters of the initiation stage (order 5). Each unit takes
the cluster of its electrode. Measure (claims file): the mean firing change over cluster A minus
over cluster B for each difference map, and the theta of cluster A minus cluster B averaged within
each sub-burst. For S2 the
clusters of the code's 94 ms initiation window fail the coherence test (ext4), so the clusters of the
250 ms window are run as a variant.

In [ ]:
rows = []
fig, axs = plt.subplots(3, 2, figsize=(10, 10))
for i, (r, stage) in enumerate([(1, 'initiation'), (3, 'initiation'), (3, 'init_250ms')]):
    sub, off = LINK[r]; e = ELEC[sub]; t0 = WIN[r]['read'][0]
    lab = LABS[(r, 5, stage)]
    ch = UCH[r]; ulab = lab[ch]
    edges = SUBB[r]
    rates = np.array([[np.sum((tr >= a * 1000) & (tr < b * 1000)) / (b - a) for a, b in zip(edges[:-1], edges[1:])] for tr in SORT[r]['trains']])
    d1, d2 = rates[:, 0] - rates[:, 1], rates[:, 1] - rates[:, 2]
    th = H.theta(WL[r], order=5)
    thA = th[:, lab == 0].mean(1); thB = th[:, lab == 1].mean(1)
    tdiff = [float(np.mean((thA - thB)[int((a - t0) * 1000):int((b - t0) * 1000)])) for a, b in zip(edges[:-1], edges[1:])]
    D1 = d1[ulab == 0].mean() - d1[ulab == 1].mean(); D2 = d2[ulab == 0].mean() - d2[ulab == 1].mean()
    flips = np.sign(D1) != np.sign(D2)
    th_alt = (np.sign(tdiff[0]) != np.sign(tdiff[1])) and (np.sign(tdiff[1]) != np.sign(tdiff[2]))
    rows.append(dict(slice=H.RECORDINGS[r]['slice'], clusters_from=stage, units_A=int((ulab == 0).sum()), units_B=int((ulab == 1).sum()),
                     active_units=[int((rates[:, k] > 0).sum()) for k in range(3)], D1_A_minus_B=D1, D2_A_minus_B=D2, firing_flips=flips,
                     theta_A_minus_B_per_subburst=np.round(tdiff, 2).tolist(), theta_alternates=th_alt, holds=bool(flips and th_alt)))
    for j, dd in enumerate((d1, d2)):
        grid = np.zeros((30, 30)); cnt = np.zeros((30, 30))
        gx = np.clip(((SORT[r]['pos'][:, 0] - e.x.min()) / (e.x.max() - e.x.min() + 1e-9) * 30).astype(int), 0, 29)
        gy = np.clip(((SORT[r]['pos'][:, 1] - e.y.min()) / (e.y.max() - e.y.min() + 1e-9) * 30).astype(int), 0, 29)
        np.add.at(grid, (gy, gx), dd); np.add.at(cnt, (gy, gx), 1)
        hm = gaussian_filter(np.where(cnt > 0, grid / np.maximum(cnt, 1), 0), 1)
        im = axs[i, j].imshow(hm, cmap='PiYG'); axs[i, j].set_title(f"{H.RECORDINGS[r]['slice']} ({stage} clusters) change {j+1} to {j+2} (range {hm.min():.0f} to {hm.max():.0f} Hz)", fontsize=8)
plt.tight_layout(); plt.show()
print(pd.DataFrame(rows).round(2).to_string(index=False))

**Result: not reproduced.** In S1 the cluster difference in firing change is 0.22 then 0.91 Hz (no flip) and the theta difference between clusters is 0.24, -0.26 and -0.65 uV over the three sub-bursts (it changes sign once). In S2 the firing difference flips (3.90 then -7.06 Hz) but the theta difference does not alternate (4.68, 2.45, -0.76 uV); with the clusters of the 250 ms window the firing difference flips (-8.92 then 5.51 Hz) and the theta difference again does not alternate (-7.20, -4.42, 1.14 uV). Neither slice meets the criterion. The S2 sub-bursts are 26 to 37 ms long, a fraction of one theta cycle.

## ext3-onset-spectral-signature

> At the onset of seizure-like activity in S1 the LFP shows a large upwelling of delta power and an
> increase in theta amplitude, and this holds at different electrodes.

0.1 to 100 Hz (order-2 high-pass at 0.1 Hz and order-5 low-pass at 100 Hz, zero phase) of the 1 kHz
LFP; complex Morlet wavelet cmor1-1 (PyWavelets), power as the squared magnitude, Gaussian smoothing
sigma 2. Onsets: the criterion's, the first burst of the first epoch, under each of the four burst
readings of fig1g; as variants, the first burst of the first epoch that has the criterion's 5 s of
recording before it, under each reading, and the initiation burst that the paper's figures and code
use (14.4 s). Baseline: the 5 s before onset, or what the recording holds of it. Electrodes: nearest
the three units with the highest rate, and three along the layer (nearest the fitted curve at 25,
50 and 75% of its x range).

In [ ]:
import pywt
s = SORT[1]; e = ELEC[LINK[1][0]]
onsets = {}
for sg, rd in READINGS:
    ep = BT[(1, sg, rd)][1]
    onsets[f'criterion ({rd}, sigma {sg})'] = float(ep[0][0])
for sg, rd in READINGS:
    ep = BT[(1, sg, rd)][1]
    onsets[f'first epoch with 5 s before ({rd}, sigma {sg})'] = float(next(x[0] for x in ep if x[0] >= 5))
onsets['paper initiation (code 14.4 s)'] = 14.4
fr = np.array([len(t) for t in s['trains']])
top = np.argsort(fr)[::-1][:3]
el_rate = [int(np.argmin((e.x - x) ** 2 + (e.y - y) ** 2)) for x, y in s['pos'][top]]
c = np.polyfit(s['pos'][:, 0], s['pos'][:, 1], 2)
xs = s['pos'][:, 0].min() + np.array([0.25, 0.5, 0.75]) * np.ptp(s['pos'][:, 0])
el_layer = [int(np.argmin((e.x - x) ** 2 + (e.y - np.polyval(c, x)) ** 2)) for x in xs]
sos_h = signal.butter(2, 0.1, 'high', fs=1000, output='sos'); sos_l = signal.butter(5, 100, 'low', fs=1000, output='sos')
freqs = np.arange(0.5, 50.5, 0.5)
scales = pywt.frequency2scale('cmor1-1', freqs / 1000)
# cmor1-1 has a Gaussian envelope exp(-t^2 / B) with B = 1 at scale s = 1 / f: at 0.5 Hz it falls to 1/e at 2 s
print('e-folding half-width of the wavelet: 0.5 Hz', 1 / 0.5, 's; 4 Hz', 1 / 4, 's')
PRE = {}
for ch in el_rate + el_layer:
    x = WL[1][:, ch].astype(float)
    x = signal.sosfiltfilt(sos_l, signal.sosfiltfilt(sos_h, x - x.mean()))
    coef, _ = pywt.cwt(x, scales, 'cmor1-1', sampling_period=1 / 1000)
    pw = gaussian_filter(np.abs(coef) ** 2, 2)
    PRE[ch] = (pw[(freqs >= 0.5) & (freqs <= 4)].mean(0), np.abs(signal.hilbert(H.theta(x))), np.abs(signal.hilbert(x)))
rows = []
for oname, on in onsets.items():
    for ename, els in [('top-rate units', el_rate), ('along layer', el_layer)]:
        for ch in els:
            delta, th_env, bb_env = PRE[ch]
            b0, b1, p1 = int(max(on - 5, 0) * 1000), int(on * 1000), int((on + 2) * 1000)
            r_delta = delta[b1:p1].mean() / delta[b0:b1].mean()
            r_th = th_env[b1:p1].mean() / th_env[b0:b1].mean()
            r_bb = bb_env[b1:p1].mean() / bb_env[b0:b1].mean()
            rows.append(dict(onset=oname, onset_s=on, baseline_s=(b1 - b0) / 1000, electrodes=ename, channel=ch, x=e.x[ch], y=e.y[ch],
                             delta_ratio=r_delta, theta_amp_ratio=r_th, broadband_amp_ratio=r_bb,
                             holds_theta=(r_delta > 1) and (r_th > 2), holds_broadband=(r_delta > 1) and (r_bb > 2)))
E3 = pd.DataFrame(rows)
pd.set_option('display.width', 250)
print(E3.round(3).to_string(index=False))
print(E3.groupby(['onset', 'electrodes'], sort=False).agg(onset_s=('onset_s', 'first'), baseline_s=('baseline_s', 'first'),
      holds_theta=('holds_theta', 'sum'), holds_broadband=('holds_broadband', 'sum')).to_string())

In [ ]:
# the spectrogram around the paper's initiation at one electrode (display)
x = WL[1][:, el_layer[1]].astype(float); x = signal.sosfiltfilt(sos_l, signal.sosfiltfilt(sos_h, x - x.mean()))
coef, _ = pywt.cwt(x[8000:22000], scales, 'cmor1-1', sampling_period=1 / 1000)
pw = gaussian_filter(np.abs(coef) ** 2, 2)
plt.figure(figsize=(10, 3)); plt.imshow(np.log10(pw), aspect='auto', origin='lower', extent=[8, 22, freqs[0], freqs[-1]]); plt.axvline(14.4, color='w')
plt.ylabel('Hz'); plt.xlabel('s'); plt.colorbar(label='log10 power'); plt.show()
print(f'log10 power range {np.log10(pw).min():.1f} to {np.log10(pw).max():.1f}')

**Result: reproduced, resting on an assumption.** The criterion's onset, the first burst of the first epoch, falls at 0.73 to 0.90 s under the four burst readings: the recording holds only 0.73 to 0.90 s of baseline, and the 2 s after onset lie within the 2 s e-folding half-width of the 0.5 Hz wavelet from the recording's start, so these runs cannot test the claim; the criterion fails there at all six electrodes (delta ratios 0.60 to 1.54, 0.1 to 100 Hz amplitude ratios 0.77 to 1.31, theta amplitude ratios 0.89 to 1.91). With onset at the paper's initiation burst (14.4 s, 5 s baseline), delta power in the 2 s after onset is 2.4 to 5.6 times the baseline at all six electrodes and the 0.1 to 100 Hz amplitude 2.09 to 2.97 times, so the criterion holds at three of three electrodes in both electrode readings with the broadband amplitude. Taking the first epoch that has 5 s of recording before it, the onset is 14.98 or 15.05 s under reading (b) and the criterion holds at three of three electrodes in both electrode readings; under reading (a) it is 13.15 or 13.76 s, isolated bursts before the initiation, and it holds at none or one. The theta amplitude never reaches twice the baseline (at most 1.92 times), so the statement's increase in theta amplitude holds only weakly.

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| table2-phase-locking-counts | not reproduced (Methods' selection; the code's selection matches S1 and S2) |
| table2-phase-locking-seizure-vs-not | partially reproduced |
| fig2b-phase-cluster-kuiper | reproduced (on the Methods' split, which is circular) |
| ext4-electrode-lag-clusters | partially reproduced |
| fig1j-theta-propagation-stages | not reproduced |
| ext4b-theta-delay | not reproduced |
| fig2a-subbursts-track-theta | not reproduced |
| ext3-onset-spectral-signature | reproduced, resting on the choice of onset |
| methods-rolling-wave-counts | blocked (methods_underspecified; the criterion's detector finds no rolling wave) |

How the analysis differs from the paper's: LFP is the 20-sample mean of the raw voltage (1 kHz), not the paper's filtered and downsampled pickles; stage windows are taken from the times in the paper's figures and code, with baselines free of detected bursts and, for S2, initiation windows of at least one theta cycle as variants; spikes for phase locking are kept while the theta envelope exceeds the RMS of theta over the analysed windows or, as a second reading, over the whole recording; the propagation direction is a plane fit to lags, a measure the claims file names, where the paper judged videos by eye; recordings 7 and 8 (S5, S6) have no raw data here, so the phase locking and electrode clustering of those slices are not tested.

What was found about the dataset: the raw files hold recordings 1 to 6 of the paper (S1 baseline from 20 s of sub-10F-0, S4 from 30 s of sub-9E-2); the S5 and S6 recordings are not in the dandiset; no file records when kainic acid was applied.

In [ ]:
POOL.shutdown()
print(f'notebook ran in {(time.time()-T0)/60:.1f} min; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss/1e6:.1f} GB')